# National overview

Questo notebook costruisce una panoramica nazionale dello stato del progetto: inventario dei dataset processati, qualita, output prodotti, registry e piano spesa sanitaria.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

for path in [Path.cwd(), *Path.cwd().parents]:
    helper_path = path / "notebooks" / "utils_notebooks.py"
    if helper_path.exists():
        sys.path.insert(0, str(helper_path.parent))
        break

from utils_notebooks import get_project_paths, plot_barh, plot_matrix, read_clean_csv, show_table

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PATHS = get_project_paths()
ROOT = PATHS["root"]
CATALOG_DIR = PATHS["catalog"]
TABLES = PATHS["tables"]
METADATA = PATHS["metadata"]
PROCESSED = PATHS["processed"]
print("Project root:", ROOT)

## Stato generale della pipeline

Partiamo dagli output generati: inventario, qualita, audit e sintesi output.

In [ ]:
inventory = read_clean_csv(TABLES / "processed_dataset_inventory.csv")
quality = read_clean_csv(TABLES / "quality_overview.csv")
outputs_summary = read_clean_csv(TABLES / "outputs_summary.csv")
project_audit = read_clean_csv(TABLES / "project_audit.csv")
source_ranking = read_clean_csv(TABLES / "source_ranking.csv")
data_requirements = read_clean_csv(TABLES / "data_requirements.csv")
module_inventory = read_clean_csv(TABLES / "module_inventory.csv")
resource_plan = read_clean_csv(TABLES / "health_expenditure_resource_plan.csv")

pipeline_summary = pd.DataFrame([
    {"area": "dataset processati inventariati", "value": len(inventory)},
    {"area": "righe con dati > 0", "value": int(pd.to_numeric(inventory.get("rows", pd.Series(dtype=float)), errors="coerce").gt(0).sum()) if not inventory.empty else 0},
    {"area": "controlli qualita", "value": len(quality)},
    {"area": "audit project fail", "value": int((project_audit.get("passed", pd.Series(dtype=str)).astype(str).str.lower() == "false").sum()) if not project_audit.empty else 0},
    {"area": "risorse piano spesa", "value": len(resource_plan)},
])
display(pipeline_summary)
show_table(inventory, 20)

## Dataset processati

L'inventario mostra quali dataset locali sono pronti e quanto sono popolati.

In [ ]:
if not inventory.empty:
    inventory["rows_numeric"] = pd.to_numeric(inventory.get("rows", 0), errors="coerce").fillna(0)
    inventory["columns_numeric"] = pd.to_numeric(inventory.get("columns", 0), errors="coerce").fillna(0)
    plot_barh(inventory, "dataset_path", "rows_numeric", "Righe per dataset processato", xlabel="Righe", color="#2563eb")
    plot_barh(inventory, "dataset_path", "columns_numeric", "Colonne per dataset processato", xlabel="Colonne", color="#0f766e")
else:
    print("Inventario dataset non disponibile.")

## Qualita degli output

I controlli separano template vuoti, errori di lettura e problemi di coerenza.

In [ ]:
if not quality.empty:
    quality["passed_label"] = quality.get("passed", "").astype(str)
    quality_counts = quality.groupby(["check_name", "passed_label"], dropna=False).size().reset_index(name="checks")
    display(quality_counts)
    plot_barh(quality_counts.groupby("passed_label")["checks"].sum().reset_index(), "passed_label", "checks", "Controlli qualita per esito", color="#b45309")
    failed = quality[quality["passed_label"].str.lower() == "false"]
    show_table(failed[[column for column in ["dataset_path", "check_name", "value", "message"] if column in failed.columns]], 30)
else:
    print("Quality overview non disponibile.")

## Output prodotti

Controllo della massa di artefatti locali: tabelle, report, figure e dashboard JSON.

In [ ]:
if not outputs_summary.empty:
    show_table(outputs_summary, 20)
    if "folder" in outputs_summary.columns:
        folder_counts = outputs_summary.groupby("folder", dropna=False).size().reset_index(name="files")
        plot_barh(folder_counts, "folder", "files", "File prodotti per cartella", color="#7c3aed")
else:
    local_files = []
    for folder_name in ["outputs/tables", "outputs/figures", "outputs/reports", "outputs/dashboard_data"]:
        folder = ROOT / folder_name
        count = len(list(folder.glob("*"))) if folder.exists() else 0
        local_files.append({"folder": folder_name, "files": count})
    plot_barh(pd.DataFrame(local_files), "folder", "files", "File prodotti per cartella", color="#7c3aed")

## Moduli e priorita fonti

Usare queste tabelle per collegare ambiti analitici, requisiti dati e fonti prioritarie.

In [ ]:
show_table(data_requirements, 30)
show_table(module_inventory, 30)

if not source_ranking.empty and "score" in source_ranking.columns:
    source_ranking["score_numeric"] = pd.to_numeric(source_ranking["score"], errors="coerce").fillna(0)
    top_sources = source_ranking.sort_values("score_numeric", ascending=False).head(12)
    show_table(top_sources[[column for column in ["provider", "source_id", "dataset_name", "theme", "score"] if column in top_sources.columns]], 12)
    plot_barh(top_sources, "source_id", "score_numeric", "Fonti nazionali piu' pronte", xlabel="Score", color="#0f766e")

## Piano spesa sanitaria

Il piano OpenBDAP/SIOPE contiene risorse candidate, formati, base contabile e score di rilevanza.

In [ ]:
if not resource_plan.empty:
    resource_plan["relevance_numeric"] = pd.to_numeric(resource_plan.get("relevance_score", 0), errors="coerce").fillna(0)
    format_counts = resource_plan.groupby("format", dropna=False).size().reset_index(name="resources") if "format" in resource_plan.columns else pd.DataFrame()
    basis_counts = resource_plan.groupby("accounting_basis", dropna=False).size().reset_index(name="resources") if "accounting_basis" in resource_plan.columns else pd.DataFrame()
    plot_barh(format_counts, "format", "resources", "Risorse spesa per formato", color="#2563eb")
    plot_barh(basis_counts, "accounting_basis", "resources", "Risorse spesa per base contabile", color="#be123c")
    top_resources = resource_plan.sort_values("relevance_numeric", ascending=False).head(20)
    show_table(top_resources[[column for column in ["provider", "source_id", "dataset_name", "format", "accounting_basis", "relevance_score", "candidate_url"] if column in top_resources.columns]], 20)
else:
    print("Piano risorse spesa sanitaria non disponibile.")

## Coda analitica nazionale

Tabella filtrabile che combina ranking fonti e piano risorse per decidere quali dataset portare in analisi.

In [ ]:
frames = []
if not source_ranking.empty:
    source_queue = source_ranking.copy()
    source_queue["queue_type"] = "source"
    source_queue["priority_score"] = pd.to_numeric(source_queue.get("score", 0), errors="coerce").fillna(0)
    source_queue["candidate_url"] = source_queue.get("source_page_url", "")
    frames.append(source_queue[[column for column in ["queue_type", "provider", "source_id", "dataset_name", "theme", "priority_score", "license", "candidate_url"] if column in source_queue.columns]])
if not resource_plan.empty:
    resource_queue = resource_plan.copy()
    resource_queue["queue_type"] = "health_expenditure_resource"
    resource_queue["priority_score"] = pd.to_numeric(resource_queue.get("relevance_score", 0), errors="coerce").fillna(0)
    frames.append(resource_queue[[column for column in ["queue_type", "provider", "source_id", "dataset_name", "theme", "priority_score", "license", "candidate_url"] if column in resource_queue.columns]])
if frames:
    analysis_queue = pd.concat(frames, ignore_index=True, sort=False)
    display(analysis_queue.sort_values("priority_score", ascending=False).head(100))
else:
    print("Nessuna coda analitica disponibile.")